In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import json
import os
from datetime import datetime

In [2]:
# Load master data
df = pd.read_csv('../data/processed/master_household_oil_impact.csv')
print(f"Loaded: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

Loaded: (69, 393)
Date range: 2020-12 to 2026-08


In [3]:
#  Temporal Features
print("CREATING TEMPORAL FEATURES...")

# Parse dates
df['date'] = pd.to_datetime(df['year_month'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['quarter'] = df['date'].dt.quarter

# Cyclical encoding for month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Seasonal flags
df['is_winter'] = df['month'].isin([12, 1, 2]).astype(int)
df['is_spring'] = df['month'].isin([3, 4, 5]).astype(int)
df['is_summer'] = df['month'].isin([6, 7, 8]).astype(int)
df['is_autumn'] = df['month'].isin([9, 10, 11]).astype(int)
df['is_heating_season'] = df['month'].isin([10, 11, 12, 1, 2, 3]).astype(int)

# Time trends
df['time_trend'] = (df['date'] - df['date'].min()).dt.days
df['months_since_start'] = ((df['date'] - df['date'].min()).dt.days / 30).astype(int)

print(f"Added {len([col for col in df.columns if col.startswith('is_') or 'sin' in col])} temporal features")

CREATING TEMPORAL FEATURES...
Added 15 temporal features


In [4]:
# Numerical Features
print("CREATING NUMERICAL FEATURES...")

#  Ratios & Proportions
df['oil_price_to_high_ratio'] = df['crude_oil_price'] / df['oil_high']
df['oil_price_to_low_ratio'] = df['crude_oil_price'] / df['oil_low']
df['oil_price_range_pct'] = (df['oil_high'] - df['oil_low']) / df['crude_oil_price'] * 100

#  Price differences
df['oil_price_diff_3m'] = df['crude_oil_price'] - df['oil_price_lag_3']
df['oil_price_diff_6m'] = df['crude_oil_price'] - df['oil_price_lag_6']
df['oil_price_diff_12m'] = df['crude_oil_price'] - df['oil_price_lag_6']  # Adjust if you have 12m lag

#  Moving averages
df['oil_price_ma_3m'] = df['crude_oil_price'].rolling(3).mean()
df['oil_price_ma_6m'] = df['crude_oil_price'].rolling(6).mean()
df['oil_price_ma_12m'] = df['crude_oil_price'].rolling(12).mean()

#  Distance from moving averages
df['oil_price_above_ma3'] = df['crude_oil_price'] - df['oil_price_ma_3m']
df['oil_price_above_ma6'] = df['crude_oil_price'] - df['oil_price_ma_6m']
df['oil_price_pct_above_ma3'] = df['oil_price_above_ma3'] / df['oil_price_ma_3m'] * 100
df['oil_price_pct_above_ma6'] = df['oil_price_above_ma6'] / df['oil_price_ma_6m'] * 100

#  Volatility features
df['oil_volatility_ma3'] = df['oil_price_volatility'].rolling(3).mean()
df['oil_volatility_ma6'] = df['oil_price_volatility'].rolling(6).mean()
df['oil_volatility_ratio'] = df['oil_price_volatility'] / df['oil_volatility_ma3']

#  Shock features
df['shock_abs'] = np.abs(df['oil_price_shock'])
df['shock_sign'] = np.sign(df['oil_price_shock'])
df['shock_squared'] = df['oil_price_shock'] ** 2
df['shock_cumulative'] = df['oil_price_shock'].cumsum()

#  Inflation-adjusted features
df['real_oil_change'] = df['inflation_adjusted_oil'] - df['inflation_adjusted_oil'].shift(1)
df['real_oil_pct_change'] = df['real_oil_change'] / df['inflation_adjusted_oil'].shift(1) * 100

print(f"Added {len([col for col in df.columns if 'ratio' in col or 'diff' in col or 'ma' in col])} numerical features")

CREATING NUMERICAL FEATURES...
Added 32 numerical features


In [5]:
#  Categorical Features
print("ENCODING CATEGORICAL FEATURES...")

# Convert categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Found {len(categorical_cols)} categorical columns")

# One-hot encoding for categoricals
for col in categorical_cols:
    if col != 'year_month':  # Keep year_month as identifier
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(columns=[col])
        print(f"  Encoded: {col} -> {len(dummies.columns)} features")

print(f"Total features after encoding: {df.shape[1]}")

ENCODING CATEGORICAL FEATURES...
Found 1 categorical columns
Total features after encoding: 428


In [6]:
# Domain-Specific Features (Your Expertise)
print("CREATING DOMAIN-SPECIFIC FEATURES...")

# Income quintile aggregates
quintiles = ['Lowest_20%', 'Second_20%', 'Middle_20%', 'Fourth_20%', 'Highest_20%']
quintile_cols = [col for col in df.columns if any(q in col for q in quintiles)]

for quintile in quintiles:
    # Get columns for this quintile
    q_cols = [col for col in quintile_cols if col.startswith(quintile) and '_10%_' not in col]
    
    if q_cols:
        # Total spend
        df[f'{quintile}_total_spend'] = df[q_cols].sum(axis=1)
        
        # Category groups
        for category, keywords in [
            ('energy', ['Electricity', 'Gas', 'Other fuels']),
            ('transport', ['Petrol', 'Diesel', 'Public_transport', 'Vehicle']),
            ('food', ['Food', 'Bread', 'Fruit', 'Vegetables', 'Milk', 'Fish', 'Poultry']),
            ('discretionary', ['Audio', 'TV', 'Holiday', 'Restaurant', 'Clothing', 'Footwear'])
        ]:
            cat_cols = [col for col in q_cols if any(k in col for k in keywords)]
            if cat_cols:
                df[f'{quintile}_{category}_spend'] = df[cat_cols].sum(axis=1)
                df[f'{quintile}_{category}_share'] = df[f'{quintile}_{category}_spend'] / df[f'{quintile}_total_spend']

# Cross-quintile comparisons
df['spend_ratio_highest_lowest'] = df['Highest_20%_total_spend'] / df['Lowest_20%_total_spend']
df['energy_share_diff'] = df['Highest_20%_energy_share'] - df['Lowest_20%_energy_share']
df['energy_share_ratio'] = df['Highest_20%_energy_share'] / df['Lowest_20%_energy_share']

# Vulnerability metrics
df['lowest_vulnerability'] = df['Lowest_20%_energy_share'] * df['oil_price_volatility']
df['highest_vulnerability'] = df['Highest_20%_energy_share'] * df['oil_price_volatility']
df['vulnerability_gap'] = df['lowest_vulnerability'] - df['highest_vulnerability']

# Oil-Expenditure interactions
df['oil_x_energy_share_lowest'] = df['crude_oil_price'] * df['Lowest_20%_energy_share']
df['oil_x_energy_share_highest'] = df['crude_oil_price'] * df['Highest_20%_energy_share']
df['shock_x_energy_share_lowest'] = df['oil_price_shock'] * df['Lowest_20%_energy_share']

print(f"Added domain-specific features")

CREATING DOMAIN-SPECIFIC FEATURES...
Added domain-specific features


In [7]:
# Save Engineered Features
print("SAVING ENGINEERED FEATURES...")

# Create directory
os.makedirs('../data/features', exist_ok=True)

# Save in multiple formats
df.to_csv('../data/features/engineered_features.csv', index=False)
df.to_parquet('../data/features/engineered_features.parquet', index=False)
df.to_pickle('../data/features/engineered_features.pkl')

# Save metadata
feature_metadata = {
    'timestamp': datetime.now().isoformat(),
    'shape': df.shape,
    'columns': df.columns.tolist(),
    'original_features': len(df.columns),
    'date_range': {
        'min': df['year_month'].min(),
        'max': df['year_month'].max()
    },
    'feature_types': {
        'temporal': [col for col in df.columns if any(x in col for x in ['year', 'month', 'quarter', 'season', 'trend'])],
        'numerical': [col for col in df.columns if any(x in col for x in ['ratio', 'diff', 'ma', 'volatility'])],
        'categorical': [col for col in df.columns if col.startswith('is_')],
        'domain': [col for col in df.columns if any(x in col for x in ['vulnerability', 'share', 'spend', 'total_'])]
    }
}

with open('../data/features/feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print(f"✅ Phase 1 Complete!")
print(f"   Engineered features: {df.shape[1]}")
print(f"   Saved to: ../data/features/")

SAVING ENGINEERED FEATURES...
✅ Phase 1 Complete!
   Engineered features: 482
   Saved to: ../data/features/
